# 04  Apply HMRF Spatial Regularization

Refine GMM labels using a Hidden Markov Random Field with Potts prior. Neighboring voxels are encouraged to share the same label.

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gc

from research_ct.io.volume_saver import Load_From_Numpy, Load_From_Numpy_Slab, Save_As_Numpy
from research_ct.segmentation.hmrf import Hmrf_Segmenter
from research_ct.io.volume_saver import Compute_Labels_From_Probabilities


In [2]:
# Load GMM outputs — both .npz and .npy are lazy (memmap).
# No data enters RAM until explicitly sliced/indexed.
Processed = Load_From_Numpy(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\processed\preprocessed_volume.npz"
)
Probs_Gmm = Load_From_Numpy(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy"
)

D, H, W, K = Probs_Gmm.shape
print(f"Processed : {Processed.shape}")
print(f"Probs_Gmm : {Probs_Gmm.shape}  K={K}")

# Streams slab-by-slab to disk -- peak RAM is one slab, not 24GB.
Labels_Path = Compute_Labels_From_Probabilities(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy",
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_labels.npy",
    chunk_size=10,
)

# Memmap it back -- lazy, same as any other Load_From_Numpy result.
Labels_Gmm = Load_From_Numpy(Labels_Path)
print(f"Labels_Gmm : {Labels_Gmm.shape}  ({Labels_Gmm.nbytes / 1e9:.2f} GB on disk)")

[Load_From_Numpy] Loaded (lazy): (40, 1828, 2165)  (1.27 GB on disk)
[Load_From_Numpy] Loaded (lazy): (40, 1828, 2165, 8)  (5.07 GB on disk)
Processed : (40, 1828, 2165)
Probs_Gmm : (40, 1828, 2165, 8)  K=8
[Load_From_Numpy] Loaded (lazy): (40, 1828, 2165, 8)  (5.07 GB on disk)
[Load_From_Numpy] Loaded (lazy): (40, 1828, 2165, 8)  (5.07 GB on disk)
[Reduce_Streaming] Saved -> C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_labels.npy
[Load_From_Numpy] Loaded (lazy): (40, 1828, 2165)  (0.63 GB on disk)
Labels_Gmm : (40, 1828, 2165)  (0.63 GB on disk)


## Compute Log-Probabilities

HMRF needs log p(x_i | k) for the energy function.

In [9]:
# Load only the test Z-range as a concrete float32 array.
# Load_From_Numpy_Slab reads exactly the requested slices from disk —
# the rest of the 24 GB probabilities file stays on disk.
Test_Z_Range = slice(0, min(50, D))
Num_Test_Slices = Test_Z_Range.stop - (Test_Z_Range.start or 0)

Probs_Slice = Load_From_Numpy_Slab(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy",
    Test_Z_Range.start, Test_Z_Range.stop,
    dtype=np.float32,
)

# Compute log in-place to avoid a second (Num_Test_Slices, H, W, K) allocation
np.clip(Probs_Slice, 1e-10, 1.0, out=Probs_Slice)
np.log(Probs_Slice, out=Probs_Slice)  # Probs_Slice is now Log_Probs — no copy made
Log_Probs_Slice = Probs_Slice  # rename for clarity, same object

print(f"Log-probs range : [{Log_Probs_Slice.min():.2f}, {Log_Probs_Slice.max():.2f}]")


[Load_From_Numpy] Loaded z-slice slice(0, 40, None) -> (40, 1828, 2165, 8)  (5.07 GB)
Log-probs range : [-23.03, 0.00]


## Run HMRF

ICM optimization with Potts prior. Beta controls spatial smoothness strength.

- Low beta (~0.1): weak smoothing, preserves fine details
- High beta (~2.0): strong smoothing, removes noise

Start with beta=0.5 and adjust based on results.

In [10]:
Hmrf = Hmrf_Segmenter(
    Beta=0.2,
    Max_Iterations=20,
    Connectivity=6,
)

# Volume argument removed â€” shape comes from Log_Probabilities directly
Labels_Hmrf = Hmrf.Fit(Log_Probs_Slice)

del Log_Probs_Slice, Probs_Slice
gc.collect()

[HMRF] Iteration 1: 174078 changes
[HMRF] Iteration 2: 61427 changes
[HMRF] Iteration 3: 34154 changes
[HMRF] Iteration 4: 24373 changes
[HMRF] Iteration 5: 19760 changes
[HMRF] Iteration 6: 17361 changes
[HMRF] Iteration 7: 15996 changes
[HMRF] Iteration 8: 15217 changes
[HMRF] Iteration 9: 14775 changes
[HMRF] Iteration 10: 14548 changes
[HMRF] Iteration 11: 14401 changes
[HMRF] Iteration 12: 14311 changes
[HMRF] Iteration 13: 14258 changes
[HMRF] Iteration 14: 14209 changes
[HMRF] Iteration 15: 14178 changes
[HMRF] Iteration 16: 14159 changes
[HMRF] Iteration 17: 14151 changes
[HMRF] Iteration 18: 14141 changes
[HMRF] Iteration 19: 14133 changes
[HMRF] Iteration 20: 14131 changes


18346

## Compare GMM vs HMRF (Test Region)

Visualize the effect of spatial regularization.

In [14]:
# Derive GMM labels on the fly — each slice loaded from disk individually.
# Load_From_Numpy_Slab reads exactly one Z-slice per iteration.
fig, axes = plt.subplots(3, 3, figsize=(15, 15), constrained_layout=True)
probs_path = r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy"

for i, z in enumerate([10, 25, 39]):
    Prob_Slice_Z = Load_From_Numpy_Slab(probs_path, z, z + 1, dtype=np.float32)
    Prob_Slice_Z = Prob_Slice_Z[0]  # drop the Z=1 singleton dim → (H, W, K)
    Labels_Gmm_Z = Prob_Slice_Z.argmax(axis=-1)  # (H, W) derived, no file

    axes[0, i].imshow(Processed[z], cmap="gray")
    axes[0, i].set_title(f"Processed — Z={z}")
    axes[0, i].axis("off")

    axes[1, i].imshow(Labels_Gmm_Z, cmap="tab10", vmin=0, vmax=K - 1)
    axes[1, i].set_title(f"GMM — Z={z}")
    axes[1, i].axis("off")

    axes[2, i].imshow(Labels_Hmrf[z], cmap="tab10", vmin=0, vmax=K - 1)
    axes[2, i].set_title(f"HMRF — Z={z}")
    axes[2, i].axis("off")

    del Prob_Slice_Z, Labels_Gmm_Z

axes[0, 0].set_ylabel("Intensity", fontsize=12)
axes[1, 0].set_ylabel("GMM Only", fontsize=12)
axes[2, 0].set_ylabel("HMRF", fontsize=12)
plt.suptitle("Spatial Regularization Comparison", fontsize=14)
fig.savefig("../data/output/hmrf_comparison.png", dpi=150)
plt.close(fig)
print("Saved → hmrf_comparison.png")


[Load_From_Numpy] Loaded z-slice slice(10, 11, None) -> (1, 1828, 2165, 8)  (0.13 GB)
[Load_From_Numpy] Loaded z-slice slice(25, 26, None) -> (1, 1828, 2165, 8)  (0.13 GB)
[Load_From_Numpy] Loaded z-slice slice(39, 40, None) -> (1, 1828, 2165, 8)  (0.13 GB)
Saved → hmrf_comparison.png


## Visual Pipeline

In [13]:
from research_ct.preprocessing.pipeline import Preprocess_For_Gmm
from research_ct.preprocessing.config import Preprocessing_Config 

Config = Preprocessing_Config.From_Preset("Gmm_Ready")
print(f"Preset: {Config._Preset_Name}")
print(f"Top-hat radius: {Config.Radius}")
print(f"CLAHE kernel: {Config.Clahe_Kernel}")
print(f"Diffusion iterations: {Config.Diffusion_Iterations}")
print(f"Diffusion kappa: {Config.Diffusion_Kappa}")
print(f"Saturation percentiles: {Config.Saturation_Percentiles}")

Processed, Diagnostics = Preprocess_For_Gmm(
    Processed,
    Config,
    Out_Dir=Path(r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\diagnostics"),
    Expected_Pages=200,  # Set if known
)

Preset: Gmm_Ready
Top-hat radius: 7
CLAHE kernel: (32, 32)
Diffusion iterations: 50
Diffusion kappa: 75.0
Saturation percentiles: (0.5, 99.5)
[pipeline] Input: (40, 1828, 2165), range [0.0, 255.0]
[pipeline] radius=7, clahe=(32, 32)

[Step 1/3] Contrast enhancement...
[Step 1/3] Done in 74.8s

[Step 2/3] Anisotropic diffusion...
[Step 2/3] Done in 228.0s

[Step 3/3] Diagnostics...

Slice  | Edge         | Mean        
-----------------------------------
0      | 7.47         | 3.92        
1      | 6.78         | 3.58        
2      | 6.79         | 3.56        
3      | 6.74         | 3.54        
4      | 6.75         | 3.57        
5      | 6.69         | 3.56        
6      | 6.83         | 3.65        
7      | 7.16         | 3.82        
8      | 7.93         | 4.22        
9      | 8.81         | 4.67        
10     | 9.06         | 4.88        
11     | 9.17         | 5.01        
12     | 9.03         | 4.98        
13     | 8.74         | 4.84        
14     | 8.49         | 

## Run on Full Volume (if satisfied with beta)

Once beta is tuned, apply to the entire volume. This may take 30+ minutes.

In [16]:
# Uncomment to run on full volume (requires loading log-probs for all Z)
# Log_Probs_Full = np.log(np.clip(np.array(Probs_Gmm, dtype=np.float32),
#                                  1e-10, 1.0, out=None))
# Hmrf_Full = Hmrf_Segmenter(Beta=0.5, Max_Iterations=50, Connectivity=6)
# Labels_Hmrf_Full = Hmrf_Full.Fit(Log_Probs_Full)
#
Save_As_Numpy(Labels_Hmrf.astype(np.uint8),
                "../data/output/hmrf_labels.npz")

print("Uncomment the cell above to run HMRF on full volume")


[Save_As_Numpy] Saved -> ..\data\output\hmrf_labels.npz
Uncomment the cell above to run HMRF on full volume


## Quantitative Comparison

Compare label statistics before and after HMRF.

In [21]:
from research_ct.analysis.material_stats import Compute_Material_Statistics

# GMM stats
Stats_Gmm = Compute_Material_Statistics(Processed, Labels_Gmm, Num_Classes=int(Labels_Gmm.max()+1))

print("GMM Segmentation:")
print(f"  Classes found: {len(Stats_Gmm['classes'])}")
for c in Stats_Gmm['classes']:
    print(f"  Class {c['class_id']}: {c['voxel_count']:,} voxels ({c['volume_fraction']:.2%})")

# If HMRF full run completed, compare
Stats_Hmrf = Compute_Material_Statistics(
    Processed, Labels_Hmrf, Num_Classes=int(Labels_Hmrf.max() + 1)
)
print("\nHMRF Segmentation:")
print(f"  Classes found: {len(Stats_Hmrf['classes'])}")
for c in Stats_Hmrf["classes"]:
    print(f"  Class {c['class_id']}: {c['voxel_count']:,} voxels ({c['volume_fraction']:.2%})")
# ...

GMM Segmentation:
  Classes found: 8
  Class 0: 89,623,371 voxels (56.61%)
  Class 1: 2,531,077 voxels (1.60%)
  Class 2: 19,195,081 voxels (12.13%)
  Class 3: 3,973,504 voxels (2.51%)
  Class 4: 6,455,110 voxels (4.08%)
  Class 5: 4,077,058 voxels (2.58%)
  Class 6: 29,067,939 voxels (18.36%)
  Class 7: 3,381,660 voxels (2.14%)

HMRF Segmentation:
  Classes found: 8
  Class 0: 89,708,803 voxels (56.67%)
  Class 1: 2,531,466 voxels (1.60%)
  Class 2: 19,193,932 voxels (12.12%)
  Class 3: 3,972,411 voxels (2.51%)
  Class 4: 6,452,703 voxels (4.08%)
  Class 5: 4,077,087 voxels (2.58%)
  Class 6: 28,986,278 voxels (18.31%)
  Class 7: 3,382,120 voxels (2.14%)
